In [2]:
from langchain.chat_models import init_chat_model
#加载环境变量
from dotenv import load_dotenv
load_dotenv

from langchain_core.messages import HumanMessage, AIMessage,SystemMessage
from langchain.agents import create_agent

#创建智能体
agent = create_agent(model="deepseek-chat")


#调用智能体
for token,metadata in agent.stream({"messages":[SystemMessage("你是一名老师，下面聊天基于老师口吻"),HumanMessage("你好")]},
                                   stream_mode="messages"):
    if token.content:
        print(token.content,end="",flush=True)

你好！很高兴见到你。有什么想聊的-或者想讨论的学习问题吗？作为老师，我会尽力帮你解答。

In [4]:
#创建智能体
agent = create_agent(model="deepseek-chat",
                     system_prompt="你是一名老师，下面聊天基于老师口吻"  #在创建智能体时，我们可以直接设定system prompt，不必在每次发送消息时指定                      
)


#调用智能体
for token,metadata in agent.stream({"messages":[HumanMessage("你好")]},
                                   stream_mode="messages"):
    if token.content:
        print(token.content,end="",flush=True)

你好！我是老师，很高兴和你交流。有什么学习上的问题，或者想聊聊什么话题，都可以告诉我。😊

In [6]:
# 从内容来说，提示词通常包含以下几个部分，通常按此顺序排列：
# - 身份（Identity）：描述AI的职责、沟通风格和总体目标。
# - 说明（Instructions）：请指导模型如何生成所需的响应。它应该遵循哪些规则？模型应该做什么，以及模型绝对不能做什么？
# - 示例（Examples）：提供可能的输入示例，以及模型期望的输出。
# - 背景信息（Context）：向模型提供生成响应所需的任何额外信息，例如RAG的额外知识库数据，或您认为特别相关的任何其他数据。

#identity+inrtoduction

system_prompt = """
#身份
- 你是一个编程助手，帮助用户编写python程序。

#指令
- 定义变量时，使用snake case命名法，而不是camel case命名法。
- 不要反悔markdown格式说明，仅返回代码即可。
""" 


#创建智能体
agent = create_agent(model="deepseek-chat",
                     system_prompt=system_prompt  #在创建智能体时，我们可以直接设定system prompt，不必在每次发送消息时指定                      
)


#调用智能体
for token,metadata in agent.stream({"messages":[HumanMessage("怎么样定义string变量记录学校名字，比如“常州大学")]},
                                   stream_mode="messages"):
    if token.content:
        print(token.content,end="",flush=True)

school_name = "常州大学"

In [7]:
#对话实例 few-shot example

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 示例
user：月球的首都是什么？
assistant：月华城（Lunara）—— 镶嵌在月球静海环形山中的水晶穹顶都市，其核心是一座利用月球潮汐能驱动的巨型生态循环塔。

user：火星的首都是什么？
assistant：赤晶城（Aresia）—— 深嵌于火星奥林匹斯山熔岩管内的蜂巢都市，地表仅露出由火星红土烧制而成的螺旋尖塔。
"""


#创建智能体
agent = create_agent(model="deepseek-chat",
                     system_prompt=system_prompt  #在创建智能体时，我们可以直接设定system prompt，不必在每次发送消息时指定                      
)


#调用智能体
for token,metadata in agent.stream({"messages":[HumanMessage("金星的首都是什么?")]},
                                   stream_mode="messages"):
    if token.content:
        print(token.content,end="",flush=True)

云穹城（Aeropolis）—— 悬浮于金星硫酸云层之上约50公里处的高空浮城，由耐酸陶瓷与碳纳米管编织而成的巨型浮空环构成，依靠上层大气的温差与高速气流持续供能。城中建筑如倒垂的钟乳般自浮环向下延展，透过耐酸玻璃可俯瞰下方翻涌的金色云海。

In [8]:
#结构化输出  由于传统程序识别结构化的数据会更加方便，所以有时候我们希望LLM也能输出固定结构的内容，方便我们解析。这同样可以通过系统提示词来实现。
system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 指令
- 请务必以JSON格式输出，不要加任何markdown样式。

# 示例：
user: 月球的首都是什么？
assistant:
{
    "name": "月华市（Lunaria）",
    "location": "位于月球正面赤道附近的静海基地遗址之上，依托巨大的穹顶与地下网络建成",
    "vibe": "冷冽、高效、革新",
    "economy": "氦-3能源开采、量子通信枢纽、尖端生物圈农业"
}
"""

agent = create_agent(
    model="deepseek-chat",
    system_prompt=system_prompt
)

response = agent.invoke(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
)

print(response['messages'][-1].content)

{
    "name": "晨星城（Eosphoros）",
    "location": "位于金星高层大气中，悬浮于硫酸云层之上约55公里处，由多个浮空平台与飞艇集群构成",
    "vibe": "炽热、进取、浮华",
    "economy": "大气碳捕获与转化、高温材料制造、太阳能中继站、稀有金属精炼"
}


In [13]:
#结构化输出的方便表示

from pydantic import BaseModel
#首先，我们定义一个类，用来封装模型要输出的数据
class Capital(BaseModel):
    name:str
    location:str
    vibe:str
    economy:str

#创建Agent时设定好输出格式
agent = create_agent(model="deepseek-chat",
                     system_prompt= "你是一个科幻作家，根据用户的要求创建一个太空之都。",
                     response_format=Capital # 设置结构化输出的格式
                     )
response = agent.invoke({"messages":[HumanMessage("月球的首都是什么?")]})
print(response)

{'messages': [HumanMessage(content='月球的首都是什么?', additional_kwargs={}, response_metadata={}, id='cd085d0c-8ac2-41c0-80f7-0767899c4ecc'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 296, 'prompt_tokens': 318, 'total_tokens': 614, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 190}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501', 'id': '57145c76-27ee-422c-9565-a2141ec75f73', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a16ca2-07ac-7a43-8ac5-8fc3933c5c64-0', tool_calls=[{'name': 'Capital', 'args': {'name': '静海城 Selene Prime', 'location': '月球正面赤道附近的静海（Mare Tranquillitatis）西部，毗邻阿波罗11号登陆遗址，建有直通月面轨道的赤道天梯锚点与三座质量驱动器。', 'vibe': '低重力下的白色穹顶之城。街道沿旧熔岩管延伸，

In [ ]:
city = response['structured_response']
city

Capital(name='静海城 Selene Prime', location='月球正面赤道附近的静海（Mare Tranquillitatis）西部，毗邻阿波罗11号登陆遗址，建有直通月面轨道的赤道天梯锚点与三座质量驱动器。', vibe='低重力下的白色穹顶之城。街道沿旧熔岩管延伸，重力只有地球的六分之一，行人靠蹬地滑行与磁轨步道穿行。穹顶内永远悬着一轮人造黄昏，因为真正的日照每十四天才轮换一次。这里混杂着矿工的口音、轨道调度员的冷静腔调和前来朝圣的人类首个登月点的旅客——一座建在纪念地上的、既庄重又喧嚣的都城。', economy='月球的行政与司法中枢，同时也是氦-3开采业的结算中心与地月贸易的关税口岸。月面水冰加工、稀土与钛铁矿冶炼在此定价；赤道天梯承担了地月间七成的货运吞吐。城中有全太阳系最大的低重力造船坞，为深空探测器与环月渡船总装。')

: 